# NHANES и Orthokeratology + Atropine: обзор файлов

Этот ноутбук нужен для первого знакомства с двумя папками данных:

- `datasets/nhanes` — открытые NHANES `.xpt` таблицы CDC;
- `datasets/Orthokeratology  0.01 Atropine` — Excel-файл исследования ортокератологии и 0.01% атропина.

Цель: понять, какие файлы есть, как они читаются, какие в них размеры/колонки и как из них собрать рабочие таблицы.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

PROJECT_DIR = Path.cwd()
NHANES_DIR = PROJECT_DIR / "datasets" / "nhanes"
ORTHO_DIR = PROJECT_DIR / "datasets" / "Orthokeratology  0.01 Atropine"

print(f"Project: {PROJECT_DIR}")
print(f"NHANES dir exists: {NHANES_DIR.exists()} -> {NHANES_DIR}")
print(f"Orthokeratology dir exists: {ORTHO_DIR.exists()} -> {ORTHO_DIR}")

Project: /home/shironoir/myopia
NHANES dir exists: True -> /home/shironoir/myopia/datasets/nhanes
Orthokeratology dir exists: True -> /home/shironoir/myopia/datasets/Orthokeratology  0.01 Atropine


## 1. Какие файлы лежат в папках

In [5]:
def list_files(folder: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            rows.append(
                {
                    "file": str(path.relative_to(PROJECT_DIR)),
                    "suffix": path.suffix.lower(),
                    "size_mb": round(path.stat().st_size / 1024 / 1024, 3),
                }
            )
    return pd.DataFrame(rows)

display(list_files(NHANES_DIR))
display(list_files(ORTHO_DIR))

,file,suffix,size_mb
0,datasets/nhanes/BMX_E.xpt,.xpt,1.717
1,datasets/nhanes/DEMO_E.xpt,.xpt,3.336
2,datasets/nhanes/DR1TOT_E.xpt,.xpt,12.237
3,datasets/nhanes/PAQ_E.xpt,.xpt,1.503
4,datasets/nhanes/VIX_E.xpt,.xpt,3.016


,file,suffix,size_mb
0,datasets/Orthokeratology 0.01 Atropine/Dataset (Kinoshita).xlsx,.xlsx,0.118


## 2. NHANES: чтение `.xpt` файлов

NHANES хранит таблицы в SAS Transport формате `.xpt`. В `pandas` они читаются через `pd.read_sas(..., format="xport")`.

Главный ключ для объединения таблиц NHANES — `SEQN`, идентификатор участника.

In [3]:
nhanes_tables = {}

for path in sorted(NHANES_DIR.glob("*.xpt")):
    name = path.stem
    df = pd.read_sas(path, format="xport")
    nhanes_tables[name] = df
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

overview_rows = []
for name, table in nhanes_tables.items():
    overview_rows.append(
        {
            "table": name,
            "rows": table.shape[0],
            "columns": table.shape[1],
            "has_SEQN": "SEQN" in table.columns,
            "first_columns": ", ".join(map(str, table.columns[:12])),
        }
    )

display(pd.DataFrame(overview_rows))

BMX_E: 9,762 rows x 23 columns
DEMO_E: 10,149 rows x 43 columns
DR1TOT_E: 9,762 rows x 164 columns
PAQ_E: 9,359 rows x 21 columns
VIX_E: 6,917 rows x 57 columns


,table,rows,columns,has_SEQN,first_columns
0,BMX_E,9762,23,True,"SEQN, BMDSTATS, BMXWT, BMIWT, BMXRECUM, BMIRECUM, BMXHEAD, BMIHEAD, BMXHT, BMIHT, BMXBMI, BMXLEG"
1,DEMO_E,10149,43,True,"SEQN, SDDSRVYR, RIDSTATR, RIDEXMON, RIAGENDR, RIDAGEYR, RIDAGEMN, RIDAGEEX, RIDRETH1, DMQMILIT, DMDBORN2, DMDCITZN"
2,DR1TOT_E,9762,164,True,"SEQN, WTDRD1, WTDR2D, DR1DRSTZ, DR1EXMER, DRABF, DRDINT, DR1DBIH, DR1DAY, DR1LANG, DR1MNRSP, DR1HELPD"
3,PAQ_E,9359,21,True,"SEQN, PAQ560, PAD590, PAD600, PAQ605, PAQ610, PAD615, PAQ620, PAQ625, PAD630, PAQ635, PAQ640"
4,VIX_E,6917,57,True,"SEQN, VIQ110, VIQ121, VIQ130, VIQ141, VIQ150, VIQ160, VIQ170, VIXNC1, VIXNC2, VIQ180, VIQ191"


Кратко по загруженным NHANES таблицам:

- `VIX_E` — vision examination: зрение, рефракция/авторефракция и связанные офтальмологические поля;
- `DEMO_E` — демография: возраст, пол, раса/этничность, веса обследования;
- `BMX_E` — body measures: рост, вес, BMI;
- `DR1TOT_E` — питание за первый 24-hour recall;
- `PAQ_E` — физическая активность.

Для анализа обычно начинают с `VIX_E`, затем присоединяют демографию и нужные covariates по `SEQN`.

In [ ]:
for name, table in nhanes_tables.items():
    print("=" * 100)
    print(name)
    display(table.head(3))
    display(table.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"}).head(25))

## 3. NHANES: пример объединения таблиц

Ниже пример рабочей таблицы: берём всех участников с vision data и присоединяем демографию, BMI, питание и физическую активность по `SEQN`.

Это ещё не финальный датасет для модели. После объединения нужно выбрать переменные, возрастную группу, определить критерий миопии и разобраться с NHANES survey weights.

In [4]:
required_tables = ["VIX_E", "DEMO_E", "BMX_E", "DR1TOT_E", "PAQ_E"]
missing_tables = [name for name in required_tables if name not in nhanes_tables]
if missing_tables:
    raise KeyError(f"Missing NHANES tables: {missing_tables}")

nhanes_merged = nhanes_tables["VIX_E"].copy()

for name in ["DEMO_E", "BMX_E", "DR1TOT_E", "PAQ_E"]:
    before_rows = len(nhanes_merged)
    nhanes_merged = nhanes_merged.merge(nhanes_tables[name], on="SEQN", how="left", suffixes=("", f"_{name}"))
    print(f"Merged {name}: {before_rows:,} -> {len(nhanes_merged):,} rows, {nhanes_merged.shape[1]:,} columns")

print(f"Final merged NHANES shape: {nhanes_merged.shape}")
display(nhanes_merged.head())

Merged DEMO_E: 6,917 -> 6,917 rows, 99 columns
Merged BMX_E: 6,917 -> 6,917 rows, 121 columns
Merged DR1TOT_E: 6,917 -> 6,917 rows, 284 columns
Merged PAQ_E: 6,917 -> 6,917 rows, 304 columns
Final merged NHANES shape: (6917, 304)


,SEQN,VIQ110,VIQ121,VIQ130,VIQ141,VIQ150,VIQ160,VIQ170,VIXNC1,VIXNC2,VIQ180,VIQ191,VIQ200,VIQ211,VIQ220,VIQ230,VIQ240,VIQ250,VIXPRS,VIXPRC,VIXPRA,VIXPLS,VIXPLC,VIXPLA,VIXOCMT,VIXORSM,VIXORCM,VIXORAM,VIDORFM,VIXOLSM,VIXOLCM,VIXOLAM,VIDOLFM,VIXKRM1,VIXKRD1,VIXKRG1,VIXKRM2,VIXKRD2,VIXKRG2,VIXKRMM,VIXKRDM,VIXKRCD,VIXKRCG,VIXKLM1,VIXKLD1,VIXKLG1,VIXKLM2,VIXKLD2,VIXKLG2,VIXKLMM,VIXKLDM,VIXKLCD,VIXKLCG,VIDRVA,VIDLVA,VIDROVA,VIDLOVA,SDDSRVYR,RIDSTATR,RIDEXMON,...,DRD370BQ,DRD370C,DRD370CQ,DRD370D,DRD370DQ,DRD370E,DRD370EQ,DRD370F,DRD370FQ,DRD370G,DRD370GQ,DRD370H,DRD370HQ,DRD370I,DRD370IQ,DRD370J,DRD370JQ,DRD370K,DRD370KQ,DRD370L,DRD370LQ,DRD370M,DRD370MQ,DRD370N,DRD370NQ,DRD370O,DRD370OQ,DRD370P,DRD370PQ,DRD370Q,DRD370QQ,DRD370R,DRD370RQ,DRD370S,DRD370SQ,DRD370T,DRD370TQ,DRD370U,DRD370UQ,DRD370V,PAQ560,PAD590,PAD600,PAQ605,PAQ610,PAD615,PAQ620,PAQ625,PAD630,PAQ635,PAQ640,PAD645,PAQ650,PAQ655,PAD660,PAQ665,PAQ670,PAD675,PAD680,PAAQUEX
0,41475.0,2.0,NaN,2.0,NaN,1.0,1.0,1.0,5.0,15.0,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000e+00,1.000000e+00,1.020000e+02,9.0,-0.75,0.75,74.0,9.0,7.85,43.00,8.0,7.65,44.00,98.0,7.75,43.50,1.00,98.0,7.81,43.25,4.0,7.62,44.25,94.0,7.72,43.75,1.00,94.0,25.0,25.0,NaN,NaN,5.0,2.0,2.0,...,2.0,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,1.0,1.0,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,720.0,1.0
1,41477.0,2.0,NaN,2.0,NaN,1.0,1.0,1.0,5.0,13.0,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.397605e-79,7.500000e-01,2.600000e+01,7.7,0.50,0.50,153.0,7.3,7.67,44.00,24.0,7.55,44.75,114.0,7.61,44.25,0.75,114.0,7.69,44.00,136.0,7.56,44.75,46.0,7.63,44.25,0.75,46.0,25.0,25.0,NaN,NaN,5.0,2.0,2.0,...,NaN,2.0,NaN,2.0,NaN,2.0,NaN,1.0,3.0,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,240.0,1.0
2,41479.0,2.0,NaN,2.0,NaN,2.0,NaN,NaN,3.0,15.0,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.397605e-79,5.397605e-79,5.397605e-79,9.0,-0.75,0.75,36.0,8.3,7.65,44.00,3.0,7.51,45.00,93.0,7.58,44.50,1.00,93.0,7.65,44.00,173.0,7.50,45.00,83.0,7.58,44.50,1.00,83.0,20.0,25.0,NaN,NaN,5.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,1.0,5.0,180.0,1.0,5.0,120.0,2.0,NaN,NaN,2.0,NaN,NaN,180.0,1.0
3,41481.0,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.500000e-01,7.500000e-01,2.100000e+01,9.0,1.75,0.25,147.0,9.0,8.00,42.25,52.0,7.87,43.00,142.0,7.94,42.50,0.75,142.0,8.01,42.25,10.0,7.93,42.50,100.0,7.97,42.25,0.25,100.0,20.0,20.0,NaN,NaN,5.0,2.0,1.0,...,NaN,1.0,1.0,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,7.0,10.0,1.0,7.0,300.0,2.0,NaN,NaN,300.0,1.0
4,41482.0,2.0,NaN,2.0,NaN,1.0,1.0,1.0,5.0,18.0,2.0,NaN,2.0,NaN,1.0,1.0,1.0,1.0,0.75,1.0,8.0,1.25,0.5,11.0,NaN,7.500000e-01,1.000000e+00,1.550000e+02,9.0,1.00,0.75,19.0,8.0,8.09,41.75,57.0,7.94,42.50,147.0,8.02,42.00,0.75,147.0,8.15,41.50,64.0,8.04,42.00,154.0,8.10,41.75,0.50,154.0,25.0,20.0,NaN,NaN,5.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,3.0,30.0,180.0,1.0


In [ ]:
if "RIDAGEYR" in nhanes_merged.columns:
    adolescents = nhanes_merged.query("12 <= RIDAGEYR <= 19").copy()
    print(f"Adolescents 12-19 with VIX_E rows: {adolescents.shape[0]:,} rows x {adolescents.shape[1]:,} columns")
    display(adolescents[["SEQN", "RIDAGEYR", "RIAGENDR"]].head())
else:
    print("RIDAGEYR not found. Check DEMO_E columns.")

Что делать с NHANES дальше:

1. Открыть codebook для `VIX_E` и выбрать переменные рефракции/остроты зрения.
2. Определить критерий миопии, например spherical equivalent refraction `<= -0.5 D`, если нужные поля доступны.
3. Ограничить возраст, например `12-19`.
4. Присоединить демографию, BMI, питание и активность.
5. Учитывать survey design и веса NHANES, если цель — популяционные выводы по США.

Ссылка на codebook vision examination: `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/VIX_E.htm`.

## 4. Orthokeratology + 0.01% Atropine: чтение Excel

В этой папке один Excel-файл. У него сложная шапка: первая строка — описание статьи, затем строка группы, затем многоуровневые заголовки. Поэтому сначала смотрим файл как raw-таблицу без заголовков.

In [5]:
ortho_files = sorted(ORTHO_DIR.glob("*.xlsx"))
if not ortho_files:
    raise FileNotFoundError(f"No .xlsx files found in {ORTHO_DIR}")

ortho_path = ortho_files[0]
print(ortho_path)

excel = pd.ExcelFile(ortho_path)
print(f"Sheets: {excel.sheet_names}")

ortho_raw = pd.read_excel(ortho_path, sheet_name=excel.sheet_names[0], header=None)
print(f"Raw shape: {ortho_raw.shape}")
display(ortho_raw.head(12))

/home/shironoir/mogt_projects/myopia_research/datasets/Orthokeratology  0.01 Atropine/Dataset (Kinoshita).xlsx
Sheets: ['OK and 0.01% atropine']
Raw shape: (154, 63)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62
0,Dataset for: Efficacy of combined orthokeratology and 0.01% atropine solution for slowing axial elongation in childr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Orthokeratology and 0.01% Atropine Combination Group,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Study ID,Institution,Age,Sex,Spherical equivalent refractive error (D),NaN,Registration date,Eye,Axial length (mm),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Corneal endothelial cell density (/mm2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Intraocular pressure (mmHg),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Uncorrected distant visual acuity (logMAR),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Uncorrected near visual acuity (logMAR),NaN,NaN,NaN,NaN,NaN,NaN,Compliance rate,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 month,NaN,3 months,3-0 months,NaN,9 months,9-3 months,NaN,15 months,15-3 months,NaN,21 months,21-3 months,NaN,27 months,27-3 months,NaN,27-15 months,NaN,0 month,NaN,3 months,15 months,15-3 months,NaN,27 months,27-3 months,NaN,0 month,NaN,3 months,15 months,15-3 months,NaN,27 months,27-3 months,NaN,0 month,NaN,3 months,15 month,15-3 months,NaN,27 months,27-3 months,NaN,3 months,15 months,15-3 months,NaN,27 months,27-3 months,NaN,OK（%）,Atropine（%）
4,1,Konno,10.589041,M,-2.25,-2.000,2015-04-10 00:00:00,R,25.54,25.450,25.52,-0.02,-0.020,25.56,0.04,0.040,25.35,-0.17,-0.060,25.71,0.19,0.195,25.68,0.16,0.175,0.33,0.235,3623,3541.5,3067,3322,255,46.5,2808,-259,-116.5,11.5,11.25,14,10.5,-3.5,-2.60,16,2,0.65,0.522879,0.610924,-0.079181,-0.079181,0,0.000000,-0.079181,0,0.000000,-0.079181,0,0.079181,0.039591,0,0.079181,0.039591,100,98
5,NaN,NaN,NaN,NaN,-1.75,NaN,NaN,L,25.36,NaN,25.34,-0.02,NaN,25.38,0.04,NaN,25.39,0.05,NaN,25.54,0.2,NaN,25.53,0.19,NaN,0.14,NaN,3460,NaN,3610,3448,-162,NaN,3636,26,NaN,11,NaN,12.7,11,-1.7,NaN,12,-0.7,NaN,0.69897,NaN,-0.079181,-0.079181,0,NaN,-0.079181,0,NaN,0,0,0,NaN,0,0,NaN,NaN,NaN
6,2,Konno,10.616438,F,-1.25,-1.125,2015-04-20 00:00:00,R,24.51,24.485,24.53,0.02,0.010,24.58,0.05,0.085,24.72,0.19,0.210,24.86,0.33,0.340,25.04,0.51,0.505,0.32,0.295,3558,3473.5,3831,3412,-419,-290.5,3472,-359,-361.5,13.5,12.75,15,11.5,-3.5,-2.50,13,-2,-2.75,0.522879,0.522879,-0.079181,0.045757,0.124939,0.062469,0.045757,0.124939,0.124939,-0.079181,-0.079181,0,0.000000,0,0.079181,0.079181,94,70
7,NaN,NaN,NaN,NaN,-1,NaN,NaN,L,24.46,NaN,24.46,0,NaN,24.58,0.12,NaN,24.69,0.23,NaN,24.81,0.35,NaN,24.96,0.5,NaN,0.27,NaN,3389,NaN,3610,3448,-162,NaN,3246,-364,NaN,12,NaN,14.5,13,-1.5,NaN,11,-3.5,NaN,0.522879,NaN,-0.079181,-0.079181,0,NaN,0.045757,0.124939,NaN,-0.079181,-0.079181,0,NaN,0,0.079181,NaN,NaN,NaN
8,3,Konno,10.687671,F,-2.75,-2.750,2015-05-25 00:00:00,R,24.95,24.890,24.88,-0.07,-0.065,25.01,0.13,0.105,25.04,0.16,0.165,25.12,0.24,0.250,25.17,0.29,0.300,0.13,0.135,3875,4012.0,3816,3412,-404,-265.0,3861,45,129.5,13,14.00,12.7,12,-0.7,0.25,12.7,0,0.20,0.823909,0.911954,-0.079181,-0.079181,0,0.000000,0,0.079181,0.039591,0,0,0,0.000000,0,0,0.000000,96,86
9,NaN,NaN,NaN,NaN,-2.75,NaN,NaN,L,24.83,NaN,24.77,-0.06,NaN,24.85,0.08,NaN,24.94,0.17,NaN,25.03,0.26,NaN,25.08,0.31,NaN,0.14,NaN,4149,NaN,3802,3676,-126,NaN,4016,214,NaN,15,NaN,12.3,13.5,1.2,NaN,12.7,0.4,NaN,1,NaN,-0.079181,-0.079181,0,NaN,-0.079181,0,NaN,0,0,0,NaN,0,0,NaN,NaN,NaN


## 5. Orthokeratology: превращаем сложную шапку в обычные колонки

В этом файле настоящие заголовки находятся в строках 2 и 3. Мы объединяем их в плоские имена вида `Axial length (mm)__0 month`, `Axial length (mm)__3 months` и т.д.

После этого можно работать с таблицей как с обычным `DataFrame`.

In [6]:
def clean_label(value) -> str:
    if pd.isna(value):
        return ""
    return " ".join(str(value).replace("\n", " ").split()).strip()


def make_unique(columns: list[str]) -> list[str]:
    seen = {}
    result = []
    for col in columns:
        base = col or "unnamed"
        seen[base] = seen.get(base, 0) + 1
        result.append(base if seen[base] == 1 else f"{base}__{seen[base]}")
    return result


header_main = ortho_raw.iloc[2].ffill().map(clean_label)
header_sub = ortho_raw.iloc[3].map(clean_label)

columns = []
for main, sub in zip(header_main, header_sub):
    if sub and sub != main:
        columns.append(f"{main}__{sub}")
    else:
        columns.append(main)

ortho = ortho_raw.iloc[4:].copy().reset_index(drop=True)
ortho.columns = make_unique(columns)
ortho = ortho.dropna(how="all")

# В Excel каждая строка часто соответствует глазу, а данные пациента указаны только на первой строке пары R/L.
# Поэтому patient-level поля протягиваем вниз.
patient_level_cols = [
    col
    for col in ortho.columns
    if col.startswith(("Study ID", "Institution", "Age", "Sex", "Registration date", "Compliance rate"))
]
ortho[patient_level_cols] = ortho[patient_level_cols].ffill()

print(f"Cleaned shape: {ortho.shape}")
display(pd.DataFrame({"column": ortho.columns}))
display(ortho.head(10))

Cleaned shape: (149, 63)


,column
0,Study ID
1,Institution
2,Age
3,Sex
4,Spherical equivalent refractive error (D)
...,...
58,Uncorrected near visual acuity (logMAR)__27 months
59,Uncorrected near visual acuity (logMAR)__27-3 months
60,Uncorrected near visual acuity (logMAR)__2
61,Compliance rate__OK（%）


,Study ID,Institution,Age,Sex,Spherical equivalent refractive error (D),Spherical equivalent refractive error (D)__2,Registration date,Eye,Axial length (mm)__0 month,Axial length (mm),Axial length (mm)__3 months,Axial length (mm)__3-0 months,Axial length (mm)__2,Axial length (mm)__9 months,Axial length (mm)__9-3 months,Axial length (mm)__3,Axial length (mm)__15 months,Axial length (mm)__15-3 months,Axial length (mm)__4,Axial length (mm)__21 months,Axial length (mm)__21-3 months,Axial length (mm)__5,Axial length (mm)__27 months,Axial length (mm)__27-3 months,Axial length (mm)__6,Axial length (mm)__27-15 months,Axial length (mm)__7,Corneal endothelial cell density (/mm2)__0 month,Corneal endothelial cell density (/mm2),Corneal endothelial cell density (/mm2)__3 months,Corneal endothelial cell density (/mm2)__15 months,Corneal endothelial cell density (/mm2)__15-3 months,Corneal endothelial cell density (/mm2)__2,Corneal endothelial cell density (/mm2)__27 months,Corneal endothelial cell density (/mm2)__27-3 months,Corneal endothelial cell density (/mm2)__3,Intraocular pressure (mmHg)__0 month,Intraocular pressure (mmHg),Intraocular pressure (mmHg)__3 months,Intraocular pressure (mmHg)__15 months,Intraocular pressure (mmHg)__15-3 months,Intraocular pressure (mmHg)__2,Intraocular pressure (mmHg)__27 months,Intraocular pressure (mmHg)__27-3 months,Intraocular pressure (mmHg)__3,Uncorrected distant visual acuity (logMAR)__0 month,Uncorrected distant visual acuity (logMAR),Uncorrected distant visual acuity (logMAR)__3 months,Uncorrected distant visual acuity (logMAR)__15 month,Uncorrected distant visual acuity (logMAR)__15-3 months,Uncorrected distant visual acuity (logMAR)__2,Uncorrected distant visual acuity (logMAR)__27 months,Uncorrected distant visual acuity (logMAR)__27-3 months,Uncorrected distant visual acuity (logMAR)__3,Uncorrected near visual acuity (logMAR)__3 months,Uncorrected near visual acuity (logMAR)__15 months,Uncorrected near visual acuity (logMAR)__15-3 months,Uncorrected near visual acuity (logMAR),Uncorrected near visual acuity (logMAR)__27 months,Uncorrected near visual acuity (logMAR)__27-3 months,Uncorrected near visual acuity (logMAR)__2,Compliance rate__OK（%）,Compliance rate__Atropine（%）
0,1,Konno,10.589041,M,-2.25,-2.000,2015-04-10 00:00:00,R,25.54,25.450,25.52,-0.02,-0.020,25.56,0.04,0.040,25.35,-0.17,-0.060,25.71,0.19,0.195,25.68,0.16,0.175,0.33,0.235,3623,3541.5,3067,3322,255,46.5,2808,-259,-116.5,11.5,11.25,14,10.5,-3.5,-2.60,16,2,0.65,0.522879,0.610924,-0.079181,-0.079181,0,0.000000,-0.079181,0,0.000000,-0.079181,0,0.079181,0.039591,0,0.079181,0.039591,100,98
1,1,Konno,10.589041,M,-1.75,NaN,2015-04-10 00:00:00,L,25.36,NaN,25.34,-0.02,NaN,25.38,0.04,NaN,25.39,0.05,NaN,25.54,0.2,NaN,25.53,0.19,NaN,0.14,NaN,3460,NaN,3610,3448,-162,NaN,3636,26,NaN,11,NaN,12.7,11,-1.7,NaN,12,-0.7,NaN,0.69897,NaN,-0.079181,-0.079181,0,NaN,-0.079181,0,NaN,0,0,0,NaN,0,0,NaN,100,98
2,2,Konno,10.616438,F,-1.25,-1.125,2015-04-20 00:00:00,R,24.51,24.485,24.53,0.02,0.010,24.58,0.05,0.085,24.72,0.19,0.210,24.86,0.33,0.340,25.04,0.51,0.505,0.32,0.295,3558,3473.5,3831,3412,-419,-290.5,3472,-359,-361.5,13.5,12.75,15,11.5,-3.5,-2.50,13,-2,-2.75,0.522879,0.522879,-0.079181,0.045757,0.124939,0.062469,0.045757,0.124939,0.124939,-0.079181,-0.079181,0,0.000000,0,0.079181,0.079181,94,70
3,2,Konno,10.616438,F,-1,NaN,2015-04-20 00:00:00,L,24.46,NaN,24.46,0,NaN,24.58,0.12,NaN,24.69,0.23,NaN,24.81,0.35,NaN,24.96,0.5,NaN,0.27,NaN,3389,NaN,3610,3448,-162,NaN,3246,-364,NaN,12,NaN,14.5,13,-1.5,NaN,11,-3.5,NaN,0.522879,NaN,-0.079181,-0.079181,0,NaN,0.045757,0.124939,NaN,-0.079181,-0.079181,0,NaN,0,0.079181,NaN,94,70
4,3,Konno,10.687671,F,-2.75,-2.750,2015-05-25 00:00:00,R,24.95,24.890,24.88,-0.07,-0.065,25.01,0.13,0.105,25.04,0.16,0.165,25.12,0.24,0.250,25.17,0.29,0.300,0.13,0.135,3875,4012.0,3816,3412,-404,-265.0,3861,45,129.5,13,14.00,12.7,12,-0.7,0.25,12.7,0,0.20,0.823909,0.911954,-0.079181,-0.079181,0,0.000000,0,0.079181,0.039591,0,0,0,0.000000,

In [ ]:
print("Orthokeratology cleaned dtypes:")
display(ortho.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"}).head(80))

missing = (
    ortho.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: (x["missing_count"] / len(ortho) * 100).round(1))
    .sort_values("missing_pct", ascending=False)
)
display(missing.head(30))

Что делать с Orthokeratology дальше:

1. Решить единицу анализа: глаз (`R`/`L`) или ребёнок. Сейчас очищенная таблица ближе к формату «одна строка = один глаз».
2. Для эффекта лечения взять признаки изменения axial length, например `Axial length (mm)__27-3 months` или похожие колонки.
3. Использовать baseline-признаки: возраст, пол, исходная рефракция, baseline axial length, compliance.
4. Для сравнения эффективности можно моделировать изменение axial length как continuous outcome, а не бинарную классификацию.
5. Если есть группы лечения в разных листах/файлах, добавить переменную `treatment_group` и сравнивать группы с поправкой на baseline.